# Tutorial 08 — Reasoning, test-time compute, and tool-using systems

[Lecture notes](../lectures/08_reasoning.md) · [Exercise sheet](../exercises/08_reasoning.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Build a schema-checked arithmetic dispatcher and compare candidate voting with an oracle-success calculation.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Validate tools before execution

No generated string is passed to eval or a shell. The dispatcher accepts exact schemas and finite numbers only.

Before running: predict the output or the invariant being checked.


In [2]:
import math

def dispatch(call):
    if not isinstance(call,dict) or set(call)!={"tool","args"}: raise ValueError("invalid call schema")
    if call["tool"] not in {"add","multiply"}: raise ValueError("unknown tool")
    args=call["args"]
    if not isinstance(args,dict) or set(args)!={"a","b"}: raise ValueError("invalid arguments")
    a,b=args["a"],args["b"]
    if any(type(x) not in (int,float) or not math.isfinite(x) or abs(x)>1e6 for x in (a,b)):
        raise ValueError("arguments must be bounded finite numbers")
    return a+b if call["tool"]=="add" else a*b
print(dispatch({"tool":"multiply","args":{"a":7,"b":8}}))
bad=[{"tool":"shell","args":{"a":1,"b":2}}, {"tool":"add","args":{"a":True,"b":2}}, {"tool":"add","args":{"a":float("nan"),"b":2}}]
for call in bad:
    try: dispatch(call)
    except ValueError as e: print("rejected",str(e))
    else: raise AssertionError("invalid call accepted")


56
rejected unknown tool
rejected arguments must be bounded finite numbers
rejected arguments must be bounded finite numbers


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. A bounded controller

This scripted controller solves (7+5)*3 in two steps. It is a test harness for action boundaries, not a learned planning model.

Before running: predict the output or the invariant being checked.


In [3]:
def run_controller(budget=2):
    trace=[]; value=None
    for step in range(budget):
        call=({"tool":"add","args":{"a":7,"b":5}} if step==0 else {"tool":"multiply","args":{"a":value,"b":3}})
        value=dispatch(call); trace.append({"call":call,"observation":value})
        if step==1: return {"answer":value,"trace":trace,"status":"complete"}
    return {"answer":None,"trace":trace,"status":"budget_exhausted"}
print(run_controller(2)); print(run_controller(1))
assert run_controller(2)["answer"]==36
assert run_controller(1)["answer"] is None


{'answer': 36, 'trace': [{'call': {'tool': 'add', 'args': {'a': 7, 'b': 5}}, 'observation': 12}, {'call': {'tool': 'multiply', 'args': {'a': 12, 'b': 3}}, 'observation': 36}], 'status': 'complete'}
{'answer': None, 'trace': [{'call': {'tool': 'add', 'args': {'a': 7, 'b': 5}}, 'observation': 12}], 'status': 'budget_exhausted'}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Voting is not oracle selection

These are fixed candidate outputs used to test aggregation. Correlated wrong answers defeat majority vote even when a correct candidate exists.

Before running: predict the output or the invariant being checked.


In [4]:
from collections import Counter
candidates=["35","35","36","35","36"]
vote=Counter(candidates).most_common(1)[0][0]
print("majority",vote,"any correct",any(x=="36" for x in candidates))
for k in [1,4,8]: print("independent oracle bound",k,1-(1-.2)**k)
assert vote!="36" and "36" in candidates


majority 35 any correct True
independent oracle bound 1 0.19999999999999996
independent oracle bound 4 0.5903999999999999
independent oracle bound 8 0.8322278399999999


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why is pass@k not the accuracy of majority voting?
2. What belongs in a tool schema besides its name?
3. Does a longer reasoning trace establish that a system is more reliable?


## Extension and submission

Use `extensions/hf_text.py` to generate candidate answers and proposed JSON calls; feed parsed proposals through the same validator, never directly into eval or a shell.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
